# Hyperboloical Compactification of 1D-Wave Equation

*This Notebook is part of the [documentation](https://anilzen.github.io/hyperboloidalFEM) on the implementation of hyperboloidal compactification methods in [NGSolve](https://ngsolve.org).*

This is a simple demonstration of a hyperboloidal layer to solve a 1D wave equation for an unknown $u$ on the unbounded domain $x\in (-\infty, \infty)$. The equation reads

$$ (-\partial_t^2 + \partial_x^2) u = 0. $$

Hyperboloidal compactification combines a hyperboloidal time transformation

$$ \tau = t + h(x), $$

with a spatial compactification

$$ \rho = g(x). $$

Define

$$ H:= \frac{dh}{dx}, \qquad G:= \frac{dg}{dx}. $$

The wave equation becomes (after division by $G$)

$$ \frac{1-H^2}{G} \partial_\tau^2 u = H \partial_\tau \partial_\rho u + \partial_\tau \partial_\rho(Hu) + \partial_\rho (G \partial_\rho u). $$

for non-smooth $h,g$ we also obtain the interface conditions at the interface $R$

$$\left(\partial_\tau Hu\right)|_{R_{outer}}+(G\partial_{\rho} u)|_{R_{outer}}=\partial_{\rho} u|_{R_{inner}}$$

where $R_{inner}$ and $R_{outer}$ are the limits from the unscaled/scaled domain.


##  weak form (2nd order)
Now multiplying the above by a test function $w$ and applying integration by parts we obtain

$$ \int_{\Omega} \frac{1-H^2}{G} \partial_\tau^2 u w = \int_\Omega  H(\partial_\tau \partial_\rho uw-\partial_\tau u\partial_\rho w) +\int_{\partial\Omega} H \partial_\tau uw - \int_\Omega G \partial_\rho u\partial_\rho w+\int_{\partial_\Omega}G \partial_\rho u\cdot n w. $$
Note that as long as G is continuous we are only left with a boundary term on the exterior (corresponding to infinity) boundary. However, as $G$ should vanish on the outer boundary only the first order in time boundary term remains.

One option now would be to discretize the above system using $H^1$ finite elements and apply a time-stepping for second order, damped equations. 

We import the necessary libraries...

In [1]:
from ngsolve import *
import ngsolve.meshes as ngm
from ngsolve.webgui import Draw

...and generate the one-dimensional mesh

In [2]:
from fem1d import geo1d

geo = geo1d(0,1,2)
geo.SetMaterials('inner','outer')
mesh = Mesh(geo.GenerateMesh(maxh=0.01))
print(mesh.GetBoundaries())
print(mesh.GetMaterials())

('left', 'right')
('inner', 'outer')


Drawing a 1d mesh is not too much fun:

In [3]:
Draw(mesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

BaseWebGuiScene

We choose the method parameters:

In [4]:
R = 1 # start of the compactification layer
rho_ = x-R # compactified coordinate (starting with 0 at R)

G = 1-rho_**2 # derivative of spatial compactification
H = -rho_ # derivative of time-domain scaling

order = 3 # polynomial order of finite element spaces

We generate the FE-spaces and bilinear Forms

In [5]:
fes = H1(mesh,order=order)
u,u_ = fes.TnT()



M_ = (
    u*u_*dx('inner')
    +(1-H**2)/G*u*u_*dx('outer')
    )

C_ = (
    H*u*grad(u_)*dx('outer')
    -H*grad(u)*u_*dx('outer')
    -H*u*u_*ds('right')
    #u*u_*dx('outer')
    )
K_ = (
    grad(u)*grad(u_)*dx('inner')
    +G*grad(u)*grad(u_)*dx('outer')
    )

We use Newmarks $\beta$ scheme for time-stepping

In [6]:
dt = 5e-4
beta = 1/4
gamma = 1/2

Sinv = BilinearForm(M_+gamma*dt*C_+beta*dt**2*K_).Assemble().mat.Inverse()
K = BilinearForm(K_).Assemble().mat
C = BilinearForm(C_).Assemble().mat
Minv = BilinearForm(M_).Assemble().mat.Inverse()

used dof inconsistency
(silence this warning by setting BilinearForm(...check_unused=False) )


In [7]:
w0 = exp(-((x-0.)**2)*30) # inital conditions

gfw = GridFunction(fes)
gfw.Set(w0)

gfw_ = GridFunction(fes)
gfw_.Set(0.)

gfw__ = GridFunction(fes)
gfw__.Set(0.)

gfw__.vec.data = -Minv@K*gfw.vec-Minv@C*gfw_.vec

tmp = gfw__.vec.CreateVector()
#scene = Draw(gfw,deformation = CF( (0,gfw,0)),min = 0, max = 0) #for drawing in loop

t=0.
tend = 1

gf_anim = GridFunction(fes,multidim = 0)


#newmark beta
while t<tend:
    tmp.data = gfw__.vec
    gfw__.vec.data = -Sinv @ K*gfw.vec-Sinv@C*gfw_.vec-dt*Sinv@K*gfw_.vec+(gamma-1)*dt*Sinv@C*tmp+(beta-1/2)*dt**2*Sinv@K*tmp
    gfw.vec.data += dt*gfw_.vec+(1/2-beta)*dt**2*tmp+beta*dt**2*gfw__.vec
    gfw_.vec.data += (1-gamma)*dt*tmp+gamma*dt*gfw__.vec
    
    t+=dt
    
    #for animation in jupyter notebook
    gf_anim.AddMultiDimComponent(gfw.vec)
    #scene.Redraw() #for redrawing

In [10]:
scene = Draw(gf_anim,deformation = CF( (0,gf_anim,0)),min = 0, max = 0, animate=True)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…